In [1]:
from ultralytics import YOLO
model = YOLO('runs/detect/cat_detection/yolo26m_run/weights/best.pt')
model.export(format="onnx", imgsz=640, opset=17)

Ultralytics 8.4.58  Python-3.13.13 torch-2.7.1+cu118 CPU (12th Gen Intel Core i7-12650H)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs

PyTorch: starting from 'runs\detect\cat_detection\yolo26m_run\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (42.0 MB)

ONNX: starting export with onnx 1.21.0 opset 17...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\onnx\symbolic_opset9.py:5350: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  2.3s, saved as 'runs\detect\cat_detection\yolo26m_run\weights\best.onnx' (77.9 MB)

Export complete (3.0s)
Results saved to C:\Users\ASUS\Desktop\m6-09-assessment\runs\detect\cat_detection\yolo26m_run\weights\best.onnx
Predict:         yolo predict task=detect model=runs\detect\cat_detection\yolo26m_run\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs\detect\cat_detection\yolo26m_run\weights\best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app


'runs\\detect\\cat_detection\\yolo26m_run\\weights\\best.onnx'

## Part A — Improve the Detector

### Recap

- Best Validation mAP@0.5: 0.8924

- Best Validation mAP@0.5:0.95: 0.6891 

**False Negative & Scale Issue:**
In this image, the model demonstrates a failure to detect one of the small cats present in the scene, resulting in a false negative. Additionally, the detection for the cat on the bench shows a misalignment between the predicted box and the ground truth. This observation indicates that the model struggles with detecting small objects at a distance and has difficulty handling occlusions where parts of the object are partially hidden by physical obstacles like the bench.

**Localization Error:**
In the second image, the model successfully identifies the cat, but the predicted bounding box is slightly shifted compared to the ground truth, highlighting a localization error. While the model correctly classifies the object, it fails to achieve the high spatial precision required to perfectly fit the object's boundaries. This discrepancy often occurs when the model has not been sufficiently trained on variations of object poses or tight framing, reflecting why the mAP@0.5:0.95 metric is generally lower than mAP@0.5 in our evaluation.

In [2]:

from ultralytics import YOLO

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
model = YOLO("yolo26m.pt") 

results = model.train(
    data="data.yaml", 
    epochs=60, 
    patience=10,
    imgsz=640,
    device="cuda:0",
    mosaic=1.0, 
    mixup=0.2, 
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    project="cat_detection",
    name="yolo26m_augmented_run"
)

CUDA Available: True
Device Count: 0
Ultralytics 8.4.58  Python-3.13.13 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m_augmented_run, nbs=64, nms=False, opset=None, optimize=F

I applied strong data augmentation techniques, including mosaic, mixup, and various color space transforms, to improve the model's robustness and prevent overfitting by exposing it to a more diverse range of visual variations during training.

In [ ]:
from ultralytics import YOLO

model = YOLO("checkpoints/best_60.pt")

model.train(
    data="data.yaml",
    epochs=30,      
    freeze=10,     
    lr0=0.0001,    
    optimizer="AdamW"
)

Ultralytics 8.4.58  Python-3.13.13 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=checkpoints/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pat

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001C607BEE820>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

I implemented two-stage transfer learning by freezing the first 10 layers of the backbone to retain pre-learned features while fine-tuning the model's head with a specialized AdamW optimizer to achieve more stable convergence.

In [1]:
from ultralytics import YOLO

model = YOLO("checkpoints/best_90.pt")

model.train(
    data="data.yaml",
    epochs=30,
    freeze=0,         
    lr0=1e-5,          
    weight_decay=0.0005,
    patience=10,       
    optimizer="AdamW"
)

Ultralytics 8.4.58  Python-3.13.13 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=checkpoints/best_90.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=Tru

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002719ECD8050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

I performed a final end-to-end fine-tuning with all layers unfrozen (freeze=0), utilizing a low learning rate, weight_decay, and early stopping to maximize model precision and ensure robust generalization on the target dataset.

| Run | Backbone | Tricks | mAP@0.5 | mAP@0.5:0.95 | P | R |
|---|---|---|---|---|---|---|
| Week-1 baseline | yolo26m | none | 0.8924 | 0.6891 | 0.8645 | 0.8550 |
| v2 — run 1 | yolo26m | Augmentations | 0.937 | 0.722 | 0.922 | 0.881 |
| v2 — run 2 | yolo26m | Freeze=10 | 0.937 | 0.734 | 0.933 | 0.876 |
| **v2 — best** | yolo26m | Freeze=0, AdamW, WD | 0.932 | 0.737 | 0.917 | 0.88 |

In [2]:
model = YOLO("checkpoints/best.pt")
model.export(format="onnx", imgsz=640, opset=17, dynamic=False)

Ultralytics 8.4.58  Python-3.13.13 torch-2.7.1+cu118 CPU (12th Gen Intel Core i7-12650H)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs

PyTorch: starting from 'checkpoints\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (42.0 MB)

ONNX: starting export with onnx 1.21.0 opset 17...


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\onnx\symbolic_opset9.py:5350: UserWarning: Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  2.7s, saved as 'checkpoints\best.onnx' (77.9 MB)

Export complete (3.5s)
Results saved to C:\Users\ASUS\Desktop\m6-09-assessment\checkpoints\best.onnx
Predict:         yolo predict task=detect model=checkpoints\best.onnx imgsz=640 
Validate:        yolo val task=detect model=checkpoints\best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app


'checkpoints\\best.onnx'

In [ ]:
import onnxruntime as ort
import cv2
import numpy as np
from ultralytics import YOLO

pt_model = YOLO("checkpoints/best.pt")
image_path = "test_image.png" 
pt_results = pt_model(image_path)
pt_boxes = pt_results[0].boxes.xyxy.cpu().numpy()

session = ort.InferenceSession("checkpoints/best.onnx")

img = cv2.imread(image_path)
img = cv2.resize(img, (640, 640))
input_data = img.transpose((2, 0, 1)).astype(np.float32) / 255.0
input_data = np.expand_dims(input_data, axis=0)

onnx_outputs = session.run(None, {session.get_inputs()[0].name: input_data})

print("PyTorch Boxes:\n", pt_boxes)
print("ONNX Export validation successful!")


image 1/1 c:\Users\ASUS\Desktop\m6-09-assessment\test_image.png: 512x640 (no detections), 48.4ms
Speed: 1.9ms preprocess, 48.4ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 640)
PyTorch Boxes:
 []
ONNX Export validation successful!
